In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

In [ ]:
project_dir = Path.cwd()
load_dotenv(project_dir / '.env')

api_key = os.getenv('OPENROUTER_API') # Set your OpenRouter API key in the .env file as OPENROUTER_API
if not api_key:
    raise ValueError("No API key found. Set OPENROUTER_API in the .env file.")

OPENROUTER_URL = 'https://openrouter.ai/api/v1/chat/completions'

# Set the model you want to use according to OpenRouter's available models. You can find the list of available models at https://openrouter.ai/models 
MODEL, MODEL_NAME = 'gpt-5.6-luna', 'openai/gpt-5.6-luna'
# MODEL, MODEL_NAME = 'qwen3.8-27b', 'qwen/qwen3.8-27b'
# MODEL, MODEL_NAME = 'llama-4-scout', 'meta-llama/llama-4-scout'

TEMPERATURE = 0

BENCHMARK = 'NextiaJD-S' # Change this to the benchmark you want to evaluate: NextiaJD-S, NextiaJD-M, OpenData, WebTable or Spider
PROMPT_PATH = project_dir / BENCHMARK / 'prompts' / 'table_JD.json'
PAIRS_PATH = project_dir / BENCHMARK / 'table_cartesians.csv'
OUTPUT_PATH = project_dir / BENCHMARK / 'output' / f'{MODEL}_results.csv'
RESUME_STATE_PATH = project_dir / BENCHMARK / 'output' / f'{MODEL}_resume_state.json'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

HEADERS = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type': 'application/json',
}


def load_resume_state() -> set[str]:
    if not RESUME_STATE_PATH.exists():
        return set()
    try:
        with RESUME_STATE_PATH.open('r', encoding='utf-8') as f:
            data = json.load(f)
        return set(data.get('processed_pairs', []))
    except (json.JSONDecodeError, TypeError, ValueError):
        return set()


def save_resume_state(processed_pairs: set[str]) -> None:
    with RESUME_STATE_PATH.open('w', encoding='utf-8') as f:
        json.dump({'processed_pairs': sorted(processed_pairs)}, f, indent=2)


processed_pairs = load_resume_state()

In [ ]:
def clean_json_response(raw_response: str) -> str:
    text = (raw_response or '').strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.S).strip()

    candidate = text
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, re.S)
        if match:
            candidate = match.group(0)
        try:
            parsed = json.loads(candidate)
        except json.JSONDecodeError:
            parsed = {
                'table_joinable': 'No',
                'column_joinable': 'No',
                'target_match': None,
                'candidate_match': None,
            }

    if not isinstance(parsed, dict):
        parsed = {
            'table_joinable': 'No',
            'column_joinable': 'No',
            'target_match': None,
            'candidate_match': None,
        }

    return json.dumps(parsed, separators=(',', ':'), ensure_ascii=False)


def safe_filename_part(value: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', '_', value).strip('._')

In [ ]:
with PROMPT_PATH.open('r', encoding='utf-8') as f:
    prompt_data = json.load(f)

pairs_df = pd.read_csv(PAIRS_PATH)
pairs_df.head()

In [ ]:
def iter_prompt_rows(prompt_data):
    for target_table, candidate_map in prompt_data.items():
        for candidate_table, messages in candidate_map.items():
            if not isinstance(messages, dict):
                raise ValueError(f'Expected a message dict for {target_table}/{candidate_table}')
            if 'system_msg' not in messages or 'user_msg' not in messages:
                raise ValueError(f'Expected system_msg and user_msg for {target_table}/{candidate_table}')
            yield {
                'LEFT_TABLE': target_table,
                'RIGHT_TABLE': candidate_table,
                'system_msg': messages['system_msg'],
                'user_msg': messages['user_msg'],
            }

prompt_rows = list(iter_prompt_rows(prompt_data))
prompt_df = pd.DataFrame(prompt_rows)
prompt_df.head()

In [ ]:
def call_openrouter_chat(model_name: str, system_msg: str, user_msg: str):
    payload = {
        'model': model_name,
        'temperature': TEMPERATURE,
        'messages': [
            {'role': 'system', 'content': system_msg},
            {'role': 'user', 'content': user_msg},
        ],
    }
    response = requests.post(
        OPENROUTER_URL,
        headers=HEADERS,
        json=payload,
        timeout=180,
    )
    response.raise_for_status()
    result = response.json()
    content = result.get('choices', [{}])[0].get('message', {}).get('content', '')
    return clean_json_response(content)

In [ ]:
processed_pairs

In [ ]:
len(processed_pairs)

In [ ]:
new_rows = []

In [ ]:
for idx, row in prompt_df.iloc[len(processed_pairs):,:].iterrows():
    left_table = row['LEFT_TABLE']
    right_table = row['RIGHT_TABLE']
    pair_key = f'{left_table}|{right_table}'
    system_msg = row['system_msg']
    user_msg = row['user_msg']

    try:
        result_json = call_openrouter_chat(MODEL_NAME, system_msg, user_msg)
    except requests.exceptions.RequestException as exc:
        print(f'Connection error while processing {pair_key}: {exc}')
        new_rows.append({
            'DATASET': BENCHMARK,
            'LEFT_TABLE': left_table,
            'RIGHT_TABLE': right_table,
            'RESPONSE': None,  
        })
        continue

    processed_pairs.add(pair_key)
    new_rows.append({
        'DATASET': BENCHMARK,
        'LEFT_TABLE': left_table,
        'RIGHT_TABLE': right_table,
        'RESPONSE': result_json,
    })

    if len(new_rows) % 100 == 0 or idx == len(prompt_df) - 1:
        df_new = pd.DataFrame(new_rows)
        df_new.to_csv(OUTPUT_PATH, mode='a', header=not OUTPUT_PATH.exists(), index=False)
        save_resume_state(processed_pairs)
        new_rows = []
        print(f'Appended through row {idx} / {len(prompt_df)-1}')

print(f'Finished. Saved new responses to {OUTPUT_PATH}')